In [29]:
import csv

# 1. Create bookings.csv
bookings_data = [
    ["movie_id", "booking_id", "show_date", "seats_booked", "show_duration_min"],
    ["M-101", "B001", "2023-10-01", 150, 120],
    ["M-101", "B002", "2023-10-01", -10, 120],  # Invalid: negative seats
    ["M-102", "B003", "2023-10-02", 250, 130],  # Overbooked (Capacity 200)
    ["M-204", "B004", "2023-10-03", 290, 150],  # High efficiency
    ["M-204", "B005", "2023-10-04", 10, 150]
]

with open("bookings.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(bookings_data)

# 2. Create movie_info.csv
movie_info_data = [
    ["movie_id", "total_seats", "genre"],
    ["M-101", 200, "Action"],
    ["M-102", 200, "Comedy"],
    ["M-204", 300, "Sci-Fi"]
]

with open("movie_info.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(movie_info_data)

print("CSV files 'bookings.csv' and 'movie_info.csv' created successfully.")

CSV files 'bookings.csv' and 'movie_info.csv' created successfully.


In [40]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *

In [41]:
spark=SparkSession.builder.appName("Movie Booking").getOrCreate()
spark.sparkContext.setLogLevel("ERROR") 

In [42]:
path="bookings.csv"
df=spark.read.csv(path,header=True,inferSchema=True)
df.show()
df.printSchema()
# df=df.withColumn("")

+--------+----------+----------+------------+-----------------+
|movie_id|booking_id| show_date|seats_booked|show_duration_min|
+--------+----------+----------+------------+-----------------+
|   M-101|      B001|2023-10-01|         150|              120|
|   M-101|      B002|2023-10-01|         -10|              120|
|   M-102|      B003|2023-10-02|         250|              130|
|   M-204|      B004|2023-10-03|         290|              150|
|   M-204|      B005|2023-10-04|          10|              150|
+--------+----------+----------+------------+-----------------+

root
 |-- movie_id: string (nullable = true)
 |-- booking_id: string (nullable = true)
 |-- show_date: date (nullable = true)
 |-- seats_booked: integer (nullable = true)
 |-- show_duration_min: integer (nullable = true)



In [43]:
def load_bookings_data(spark: SparkSession, path: str) -> DataFrame:
    df = spark.read.csv(path, header=True, inferSchema=True)
    # Cast show_date to DateType
    df = df.withColumn("show_date", to_date(col("show_date"), "yyyy-MM-dd"))
    return df

In [44]:
def load_movie_info(spark: SparkSession, path: str) -> DataFrame:
    return spark.read.csv(path, header=True, inferSchema=True)

In [45]:
def filter_valid_bookings(df: DataFrame) -> DataFrame:
    return df.filter((col("seats_booked") >= 0) & (col("show_duration_min") >= 0))

In [46]:
def join_movie_info(bookings_df: DataFrame, info_df: DataFrame) -> DataFrame:
    return bookings_df.join(info_df, on="movie_id", how="left")

In [47]:
def with_overbooking_flag(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "overbooked", 
        when(col("seats_booked") > col("total_seats"), 1).otherwise(0)
    )

In [50]:
def movie_occupancy_efficiency(df: DataFrame) -> str:
    # Group by movie_id and total_seats
    agg_df = df.groupBy("movie_id", "total_seats") \
               .agg(sum("seats_booked").alias("total_booked"))
    
    # Compute efficiency using guarded division
    efficiency_df = agg_df.withColumn(
        "efficiency",
        when(col("total_seats") == 0, 0.0).otherwise(col("total_booked") / col("total_seats"))
    )
    
    # Sort by efficiency descending
    sorted_df = efficiency_df.orderBy(col("efficiency").desc())
    
    # Return top movie_id as string (guard against empty dataframe)
    top_row = sorted_df.select("movie_id").first()
    return top_row["movie_id"] if top_row else ""

In [51]:
if __name__ == "__main__":
    # 1 & 2. Load Data
    bookings_df = load_bookings_data(spark, "bookings.csv")
    movie_info_df = load_movie_info(spark, "movie_info.csv")
    
    # 3. Filter valid bookings
    valid_bookings_df = filter_valid_bookings(bookings_df)
    
    # 4. Join datasets
    joined_df = join_movie_info(valid_bookings_df, movie_info_df)
    
    # 5. Add overbooked flag
    enriched_df = with_overbooking_flag(joined_df)
    
    print("--- Enriched DataFrame ---")
    enriched_df.show()
    
    # 6. Calculate efficiency and get the top movie
    top_movie = movie_occupancy_efficiency(enriched_df)
    
    print(f"Movie with the highest occupancy efficiency: {top_movie}")

--- Enriched DataFrame ---
+--------+----------+----------+------------+-----------------+-----------+------+----------+
|movie_id|booking_id| show_date|seats_booked|show_duration_min|total_seats| genre|overbooked|
+--------+----------+----------+------------+-----------------+-----------+------+----------+
|   M-101|      B001|2023-10-01|         150|              120|        200|Action|         0|
|   M-102|      B003|2023-10-02|         250|              130|        200|Comedy|         1|
|   M-204|      B004|2023-10-03|         290|              150|        300|Sci-Fi|         0|
|   M-204|      B005|2023-10-04|          10|              150|        300|Sci-Fi|         0|
+--------+----------+----------+------------+-----------------+-----------+------+----------+

Movie with the highest occupancy efficiency: M-102


In [52]:
spark.stop()

In [ ]:
df = df.withColumn("price", col("price").cast("decimal(10,2)"))

In [20]:
data = [
    ("2026-01-15", "2026-01-15 14:30:00", "15/01/2026"),
    ("2026-12-31", "2026-12-31 23:59:59", "31/12/2026"),
]

# Standard .cast() works perfectly IF strings are in ISO format (yyyy-MM-dd HH:mm:ss)
df_cleaned = df \
    .withColumn("pure_date", col("standard_date_str").cast("date")) \
    .withColumn("pure_timestamp", col("standard_ts_str").cast("timestamp"))
 
# For custom formats, use to_date() or to_timestamp() instead of .cast().
df_final = df_cleaned.withColumn(
    "custom_parsed_date", 
    to_date(col("custom_date_str"), "dd/mm/yyyy")
)

--- Original Schema ---
root
 |-- standard_date_str: string (nullable = true)
 |-- standard_ts_str: string (nullable = true)
 |-- custom_date_str: string (nullable = true)

--- Transformed Schema & Data ---
root
 |-- standard_date_str: string (nullable = true)
 |-- standard_ts_str: string (nullable = true)
 |-- custom_date_str: string (nullable = true)
 |-- pure_date: date (nullable = true)
 |-- pure_timestamp: timestamp (nullable = true)
 |-- custom_parsed_date: date (nullable = true)

+----------+-------------------+------------------+
|pure_date |pure_timestamp     |custom_parsed_date|
+----------+-------------------+------------------+
|2026-01-15|2026-01-15 14:30:00|2026-01-15        |
|2026-12-31|2026-12-31 23:59:59|2026-01-31        |
+----------+-------------------+------------------+

